# Priorización Territorial - Análisis de Clustering Jerárquico y PCA


### 1. Importación de Librerías

In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.stats import ttest_ind
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

### 2. Definición de Funciones Core
Definición de funciones de procesamiento matemático, PCA, clustering y visualización.

In [ ]:
def preprocess_data(data, variables, variance_threshold=0.8):
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data[variables])
    pca = PCA()
    pca_data = pca.fit_transform(standardized_data)
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    n_components_used = np.argmax(cumulative_variance >= variance_threshold) + 1

    puntuaciones_pca = pca_data[:, :n_components_used]

    puntuaciones_normalizadas = np.apply_along_axis(
        lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)), axis=0, arr=puntuaciones_pca
    )
    kpi_por_registro = np.sum(puntuaciones_normalizadas, axis=1)

    quintiles = pd.qcut(kpi_por_registro, q=5, labels=["Muy Bajo", "Bajo", "Medio", "Alto", "Muy Alto"])

    return puntuaciones_pca, n_components_used, cumulative_variance[n_components_used - 1] * 100, kpi_por_registro, quintiles, pca


def hierarchical_clustering(data, num_clusters, method, metric):
    linkage_matrix = linkage(data, method=method, metric=metric)
    clusters = fcluster(linkage_matrix, num_clusters, criterion='maxclust')
    return clusters, linkage_matrix


def evaluate_clustering(data, clusters):
    silhouette_avg = silhouette_score(data, clusters)
    cluster_sizes = pd.Series(clusters).value_counts()
    balance_score = 1 / (1 + cluster_sizes.std())
    min_cluster_proportion = cluster_sizes.min() / len(clusters)
    size_penalty = min_cluster_proportion if min_cluster_proportion > 0.05 else 0 

    combined_score = silhouette_avg * balance_score * size_penalty
    return combined_score, silhouette_avg, balance_score, size_penalty


def find_best_clustering(data, num_clusters):
    methods = ['ward', 'single', 'complete', 'average', 'weighted', 'centroid', 'median']
    metrics = ['euclidean', 'cityblock', 'cosine', 'correlation', 'canberra', 'chebyshev']
    best_combined_score = -1
    best_params = None

    for method in methods:
        for metric in metrics:
            if method == 'ward' and metric != 'euclidean':
                continue
            try:
                clusters, linkage_matrix = hierarchical_clustering(data, num_clusters, method, metric)
                unique_clusters = len(np.unique(clusters))
                
                if unique_clusters == num_clusters:
                    combined_score, silhouette_avg, balance_score, size_penalty = evaluate_clustering(data, clusters)
                    if combined_score > best_combined_score:
                        best_combined_score = combined_score
                        best_params = {
                            'method': method,
                            'metric': metric,
                            'clusters': clusters,
                            'linkage_matrix': linkage_matrix,
                            'silhouette': silhouette_avg,
                            'balance': balance_score,
                            'penalty': size_penalty
                        }
            except Exception:
                continue

    if best_params:
        return (
            best_params['method'],
            best_params['metric'],
            best_combined_score,
            best_params['silhouette'],
            best_params['balance'],
            best_params['penalty'],
            best_params['clusters'],
            best_params['linkage_matrix']
        )
    else:
        print('No se pudo encontrar una solución con el número exacto de clusters.')
        return None, None, None, None, None, None, None, None


def create_cluster_summary(original_data, clusters, variables):
    original_data['Cluster'] = clusters
    variables_with_kpi = variables + ['KPI'] if 'KPI' not in variables else variables

    summary = original_data.groupby('Cluster')[variables_with_kpi].mean().T
    summary.columns = [f'Cluster {col}' for col in summary.columns]
    
    obs_counts = original_data['Cluster'].value_counts().sort_index()
    summary['Observaciones'] = None
    
    for cluster in obs_counts.index:
        cluster_col = f'Cluster {cluster}'
        summary.loc['Observaciones', cluster_col] = obs_counts[cluster]
    
    return summary.reset_index().rename(columns={'index': 'Variable'})


def color_summary_with_significance(summary, original_data, variables):
    cluster_cols = [col for col in summary.columns if col.startswith('Cluster')]
    styled_summary = summary.copy()

    for var in variables:
        if var in summary['Variable'].values:
            row_index = summary.index[summary['Variable'] == var][0]
            means = summary.loc[row_index, cluster_cols].values
            max_val = max([x for x in means if x is not None])
            min_val = min([x for x in means if x is not None])

            significant = False
            for i in range(len(cluster_cols)):
                for j in range(i + 1, len(cluster_cols)):
                    group1 = original_data[original_data['Cluster'] == i+1][var].dropna()
                    group2 = original_data[original_data['Cluster'] == j+1][var].dropna()
                    if len(group1) > 1 and len(group2) > 1:
                        _, p_value = ttest_ind(group1, group2, equal_var=False)
                        if p_value < 0.05:
                            significant = True
                            break
                if significant:
                    break

            if significant:
                for col in cluster_cols:
                    val = summary.loc[row_index, col]
                    if val == max_val:
                        styled_summary.at[row_index, col] = f"<span style='color:green; font-weight:bold;'>{val:.2f}</span>"
                    elif val == min_val:
                        styled_summary.at[row_index, col] = f"<span style='color:red; font-weight:bold;'>{val:.2f}</span>"
                    else:
                        styled_summary.at[row_index, col] = f'{val:.2f}'

    return styled_summary


def plot_perceptual_map_vectors_only(pca, variables):
    fig, ax = plt.subplots(figsize=(8, 8))
    for i, variable in enumerate(variables):
        x = pca.components_[0, i]
        y = pca.components_[1, i]
        ax.arrow(0, 0, x, y, color='r', alpha=0.7, head_width=0.02)
        ax.text(x * 1.1, y * 1.1, variable, color='r', ha='center', va='center')

    ax.set_title('Mapa Perceptual - Vectores de Atributos')
    ax.set_xlabel('Componente Principal 1')
    ax.set_ylabel('Componente Principal 2')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.grid(True)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    plt.tight_layout()
    return fig

### 3. Carga de Datos

In [ ]:
file_path = 'consolidado.xlsx'
data = pd.read_excel(file_path)
data['MpCodigo'] = data['MpCodigo'].astype(str)


print('Primeras filas del conjunto de datos:')
display(data.head())

### 4. Configuración de Parámetros

In [ ]:
variables_seleccionadas = ['Estimación Inseguridad Alimentaria Moderada-Grave',	'IRCA',	'Suficiencia de vías',	'Cantidad tipo de cultivos',	'temperature_2m_mean_c',	'NDVI',	'Pob rural']
num_clusters = 3

manual_selection = False
linkage_method = 'ward'
metric = 'euclidean'

### 5. Ejecución

In [ ]:
pca_data, n_components_used, var_explained, kpi, quintiles, pca = preprocess_data(data, variables_seleccionadas)

if manual_selection:
    clusters, linkage_matrix = hierarchical_clustering(pca_data, num_clusters, linkage_method, metric)
    best_method = linkage_method
    best_metric = metric
else:
    best_method, best_metric, best_combined_score, best_silhouette, best_balance, best_penalty, clusters, linkage_matrix = find_best_clustering(
        pca_data, num_clusters
    )

data['KPI'] = kpi
data['KPI_Quintile'] = quintiles
data['Cluster'] = clusters

print('=' * 40)
print('MÉTRICAS Y RESULTADOS DEL MODELO')
print('=' * 40)
print(f'Método de enlace óptimo: {best_method}')
print(f'Métrica de distancia: {best_metric}')
print(f'Componentes principales retenidos: {n_components_used}')
print(f'Varianza explicada total: {var_explained:.2f}%')
print('=' * 40)

### 6. Visualizaciones

In [ ]:
fig_map = plot_perceptual_map_vectors_only(pca, variables_seleccionadas)
plt.show()

plt.figure(figsize=(10, 5))
dendrogram(linkage_matrix, color_threshold=linkage_matrix[-num_clusters + 1, 2], above_threshold_color='k')
plt.title('Dendrograma de la Estructura Jerárquica')
plt.xlabel('Índice de las Observaciones')
plt.ylabel('Distancia Linkage')
plt.axhline(y=linkage_matrix[-num_clusters + 1, 2], color='r', linestyle='--')
plt.tight_layout()
plt.show()

### 7. Resumen

In [ ]:
summary = create_cluster_summary(data.copy(), clusters, variables_seleccionadas)
styled_summary = color_summary_with_significance(summary, data, variables_seleccionadas)

print('Resumen descriptivo de medias por variable (Verde: Máximo significativo | Rojo: Mínimo significativo):')
display(HTML(styled_summary.to_html(escape=False, index=False)))

output_filename = 'resultados_priorizacion_territorial.xlsx'
selected_columns = ['MpCodigo'] + variables_seleccionadas + ['KPI', 'KPI_Quintile', 'Cluster']
data[selected_columns].to_excel(output_filename, index=False)
print(f'\n💾 Base de datos procesada guardada como {output_filename}')